# Earnalism music job `music-film-01`

**Prepared, not run.** Input zip SHA256 `fc1627594b74358d428713ed20c602a8517ee1263ea6fab23462dea87623c2f3`. Copy `inputs.zip` to `/content/drive/MyDrive/earnalism-audiovisual/bicharak/music-film-01/` in Drive, set `RUN = True` in the first cell, then run all. Outputs and `manifest.json` (sha256 of every output) land in the same folder.


In [ ]:
RUN = False

In [ ]:
if RUN is not True:
    raise RuntimeError("RUN is False; set RUN = True in the first cell to run this job")
import pathlib
JOB_ID = 'music-film-01'
TITLE = 'bicharak'
KIND = 'music'
INPUT_ZIP_SHA256 = 'fc1627594b74358d428713ed20c602a8517ee1263ea6fab23462dea87623c2f3'
MODEL_REPO = 'ACE-Step/ACE-Step-v1-3.5B'
MODEL_REVISION = 'resolved-at-run-time'
STEPS = 60
DRIVE_JOB_DIR = '/content/drive/MyDrive/earnalism-audiovisual/bicharak/music-film-01'


In [ ]:
if RUN is not True:
    raise RuntimeError("RUN is False; set RUN = True in the first cell to run this job")
import hashlib, json, pathlib, zipfile
from google.colab import drive
drive.mount("/content/drive")
OUT_DIR = pathlib.Path(DRIVE_JOB_DIR)
INPUT_ZIP = OUT_DIR / "inputs.zip"
if not INPUT_ZIP.is_file():
    raise RuntimeError("Copy inputs.zip into " + str(OUT_DIR) + " first")
actual = hashlib.sha256(INPUT_ZIP.read_bytes()).hexdigest()
if actual != INPUT_ZIP_SHA256:
    raise RuntimeError("inputs.zip sha256 mismatch: " + actual)
with zipfile.ZipFile(INPUT_ZIP) as z:
    JOB = json.loads(z.read("job.json"))
if JOB["job_id"] != JOB_ID:
    raise RuntimeError("job.json belongs to a different job")
print("inputs verified for", JOB_ID)


In [ ]:
# Music: ACE-Step. The exact commit and weight hashes are resolved now and recorded.
if RUN is not True:
    raise RuntimeError("RUN is False; set RUN = True in the first cell to run this job")
import hashlib, subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "git+https://github.com/ace-step/ACE-Step.git"], check=True)
from huggingface_hub import HfApi, snapshot_download

revision = HfApi().model_info(MODEL_REPO).sha
weights_dir = snapshot_download(MODEL_REPO, revision=revision)


def _sha(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


weight_hashes = {}
for p in sorted(pathlib.Path(weights_dir).rglob("*")):
    if p.is_file() and p.suffix in (".safetensors", ".bin", ".pt", ".pth"):
        weight_hashes[p.relative_to(weights_dir).as_posix()] = _sha(p)
MODEL_INFO = {"repo": MODEL_REPO, "revision": revision, "weight_sha256": weight_hashes}

from acestep.pipeline_ace_step import ACEStepPipeline

pipe = ACEStepPipeline(checkpoint_dir=weights_dir, dtype="bfloat16")
for t in JOB["items"]:
    pipe(prompt=t["prompt"], lyrics=t.get("lyrics", "[instrumental]"),
         audio_duration=t["duration_s"], infer_step=60, guidance_scale=15.0,
         manual_seeds=[t["seed"]], save_path=str(OUT_DIR / (t["key"] + ".wav")))
    print("wrote", t["key"])


In [ ]:
if RUN is not True:
    raise RuntimeError("RUN is False; set RUN = True in the first cell to run this job")
import hashlib, json
outputs = {}
for p in sorted(OUT_DIR.iterdir()):
    if p.is_file() and p.name not in ("manifest.json", "inputs.zip"):
        outputs[p.name] = hashlib.sha256(p.read_bytes()).hexdigest()
if not outputs:
    raise RuntimeError("no outputs were produced")
manifest = {"schema": "earnalism.audiovisual-outputs.v1", "job_id": JOB_ID, "title": TITLE,
            "kind": KIND, "input_zip_sha256": INPUT_ZIP_SHA256, "model": MODEL_INFO,
            "outputs": outputs}
(OUT_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2, sort_keys=True) + "\n")
print("manifest.json written:", len(outputs), "files")
